In [1]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

import spacy
from rank_bm25 import BM25Okapi

import pathlib, os

import pandas as pd

from tqdm import tqdm

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
def data_preparation(dataset:str):
       
       # Download dataset and unzip the dataset
       url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
       out_dir = os.path.join(pathlib.Path(os.path.abspath('')), "datasets")
       data_path = util.download_and_unzip(url, out_dir)
       
       # train/test split
       d_train,q_train,qr_train=GenericDataLoader(data_path).load(split="train")
       _,q_test,qr_test=GenericDataLoader(data_path).load(split="test")
       
       # to pandas dataframe
       train={"documents":pd.DataFrame.from_dict(d_train,orient="index").drop(["title"], axis=1),
              "queries":pd.DataFrame.from_dict(q_train,orient="index",columns=["text"]),
              "real":qr_train}
       
       test={"queries":pd.DataFrame.from_dict(q_test,orient="index",columns=["text"]),
              "real":qr_test}
              

       
       def data_tokenizer(train:dict[pd.DataFrame],test:dict[pd.DataFrame]):
              nlp = spacy.load("en_core_web_sm", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer", "ner"])
              tqdm.pandas()
              
              def tokenizer(text):
                     return " ".join(token.text for token in nlp(text) if not token.is_stop and not token.is_punct)
              
              train["documents"]["text"]=train["documents"]["text"].progress_apply(tokenizer)
              train["queries"]["text"]=train["queries"]["text"].progress_apply(tokenizer)
              
              #test["documents"]["text"]=test["documents"]["text"].progress_apply(tokenizer)
              test["queries"]["text"]=test["queries"]["text"].progress_apply(tokenizer)
       
       
       #print(train["documents"].equals(test["documents"]))
       print(train["queries"].equals(test["queries"]))
       
       #data tokenization using spacy
       data_tokenizer(train,test)
       
       return train,test

train,test=data_preparation("fiqa")

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

False


100%|██████████| 648/648 [00:00<00:00, 22343.88it/s]


In [5]:
from concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor,as_completed

import multiprocessing
import heapq
from threading import Lock

bar=tqdm(total=len(train["queries"]),desc=str("ciao"))

def update_wrapper():
    bar.update()
    #bar.display()

def prova(start,stop,lock):
    
    bm25 = BM25Okapi(train["documents"]["text"].tolist())
    
    scores_per_query={}

    
    for index,row in train["queries"].iloc[start:stop].iterrows():
        
        scores=bm25.get_scores(row["text"])
        
        scores_per_query[index]=scores
        
        with lock:
            print("sequential")
            update_wrapper()
        
        #print(row["text"],bm25.get_top_n(row["text"], train["documents"]["text"].tolist(), n=1))
    
    
    return scores_per_query


    
with ProcessPoolExecutor(max_workers=8) as executor:
    
    bar.display()
    
    length=int(len(train["queries"])/8)
    
    futures=[]
    
    lock = multiprocessing.Manager().Lock()
        
    for pos,start in enumerate(range(0,len(train["queries"]),length)):
        
        futures.append( executor.submit(prova,start,start+length,lock) )
    
    for future in as_completed(futures):
        print(future.result())


ciao:   1%|          | 64/5500 [01:27<1:56:26,  1.29s/it]

In [ ]:
scores_per_query

{'0': array([187.97826208, 184.71644754, 182.8035483 , ..., 187.09430487,
        185.64929251, 187.20445658]),
 '4': array([294.88403529, 299.491401  , 290.8265135 , ..., 295.39004255,
        296.30945469, 292.47828366]),
 '5': array([126.37618207, 125.42854532, 116.00204627, ..., 125.11342737,
        122.83769918, 124.15952231]),
 '6': array([101.99924913, 101.59418356, 100.27618466, ..., 101.24217489,
        100.24497878,  99.98444957]),
 '7': array([230.67934578, 225.94654156, 212.0884998 , ..., 230.19551175,
        218.46818756, 229.99727186]),
 '9': array([69.30749663, 67.30468144, 67.09264784, ..., 67.11031305,
        62.32383874, 68.89455368]),
 '11': array([165.95301444, 162.37299556, 159.92196018, ..., 166.20734858,
        163.31515944, 164.82935049]),
 '12': array([335.3002934 , 321.12548085, 323.73496787, ..., 330.80090501,
        324.27326409, 332.73968862]),
 '13': array([209.40113687, 207.29587399, 196.30674341, ..., 206.50244254,
        201.31817719, 205.7003249

In [ ]:
train["documents"]["text"].tolist()

['saying like idea job training expect company Training workers job building software educational systems U.S. students worry little getting marketable skills exchange massive investment education getting thousands student debt complaining qualified',
 'preventing false ratings additional scrutiny market investors newer controls place prevent institutions DFA banks longer solely rely credit ratings diligence buy financial instrument plus intent financial institutions leg work maybe figure certain CDO garbage   Edit lead',
 'use health FSA individual health insurance premiums   FSA plan sponsors limit reimburse   use health FSA premiums previously use 125 cafeteria plan pay premiums separate election health FSA N. 2013 54 cafeteria plan pay indivdiual premiums effectively prohibited',
 'Samsung created LCD flat screen technology like OLED years ago flat screen came Samsung factories reshelled think 21 Hanns screen looking Samsung couple years old Samsung good company',
 'SEC requirement